# 演習④ 評価指標を自分で計算する（穴埋め形式）

**対応ノートブック**: `notebooks/ch2_04_evaluation.ipynb`
**推奨所要時間**: 約 30 分

`# TODO:` と書かれた箇所を自分で埋めてください。
各課題の最後には **自己チェック**（`assert`）が入っています。`✅ OK` が出れば正解です。

この演習では、1 つの模範解答に対する 4 つの回答（完全一致・言い換え・不完全・正反対）を、3 つの指標で採点します。
「正反対」は、否定の語を足しただけで意味が逆になった回答です。指標がこれを見抜けるかに注目してください。

---

## セットアップ（完成済み）

In [ ]:
# 実行時間: 数秒
import os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer
from bert_score import score as bert_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用デバイス: {device}')

reference = '機械学習はデータからパターンを学習し、新しいデータに対して予測を行う技術です。'
hypotheses = {
    '完全一致': '機械学習はデータからパターンを学習し、新しいデータに対して予測を行う技術です。',
    '言い換え': 'データに潜む規則性を見つけ出して、未知の入力の結果を推測する手法が機械学習です。',
    '不完全':   '機械学習はデータを使う技術です。',
    '正反対':   '機械学習はデータからパターンを学習せず、新しいデータに対して予測を行わない技術です。',
}
for name, h in hypotheses.items():
    print(f'{name:<6}: {h}')

---

## 課題 1: ROUGE-L を自分で実装する

文字単位の ROUGE-L（F 値）を返す関数を完成させてください。

1. 2 つの文字列の **最長共通部分列（LCS）** の長さを、動的計画法で求める
   - `dp[i][j]` = `a` の先頭 i 文字と `b` の先頭 j 文字の LCS の長さ
   - `a[i-1] == b[j-1]` なら `dp[i-1][j-1] + 1`、違えば `max(dp[i-1][j], dp[i][j-1])`
2. Precision = LCS ÷ 回答の文字数、Recall = LCS ÷ 模範解答の文字数、F = 2PR ÷ (P + R)

空白は数えません（`rouge_score` に渡す `CharTokenizer` と同じ扱い）。

In [ ]:
def lcs_length(a: str, b: str) -> int:
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            # TODO: dp[i][j] を埋めてください
            pass
    return dp[len(a)][len(b)]


def rouge_l(reference: str, hypothesis: str) -> float:
    ref = ''.join(reference.split())     # 空白を除く
    hyp = ''.join(hypothesis.split())
    lcs = lcs_length(ref, hyp)
    if lcs == 0:
        return 0.0
    # TODO: Precision・Recall・F 値を計算して、F 値を返してください
    return None


rouge_scores = {name: rouge_l(reference, h) for name, h in hypotheses.items()}
for name, s in rouge_scores.items():
    print(f'{name:<6}: ROUGE-L = {s:.3f}')

# ── 自己チェック（rouge_score に文字単位のトークナイザを渡した結果と比べる）──
class CharTokenizer:
    def tokenize(self, text):
        return [c for c in text if not c.isspace()]

scorer = rouge_scorer.RougeScorer(['rougeL'], tokenizer=CharTokenizer())
for name, h in hypotheses.items():
    expected = scorer.score(reference, h)['rougeL'].fmeasure
    assert abs(rouge_scores[name] - expected) < 1e-9, f'{name}: {rouge_scores[name]} と {expected} が一致しません'
assert lcs_length('ABCBDAB', 'BDCABA') == 4
print('✅ OK')

---

## 課題 2: BERTScore を計算する

`bert_score()` で、4 つの回答の BERTScore（F1）を計算してください。

- `model_type='bert-base-multilingual-cased'` を指定する（`lang='ja'` でも同じモデルが選ばれるが、どのモデルで測ったかが分かるよう明示する）
- `bert_score(回答のリスト, 模範解答のリスト, model_type=..., device=device)` は `(P, R, F1)` を返す。F1 は 1 件ずつの値が入ったテンソル

In [ ]:
names = list(hypotheses)

# TODO: 4 つの回答の BERTScore F1 を計算し、{名前: F1} の辞書にしてください
bert_scores = None

for name in names:
    print(f'{name:<6}: ROUGE-L = {rouge_scores[name]:.3f}  BERTScore = {bert_scores[name]:.3f}')

# ── 自己チェック ──
assert set(bert_scores) == set(hypotheses), '4 つすべての回答を計算してください'
assert abs(bert_scores['完全一致'] - 1.0) < 1e-3, '同じ文どうしの BERTScore は 1.0 になるはずです'
assert all(0 < v <= 1.0 + 1e-6 for v in bert_scores.values())
print('✅ OK')

---

## 課題 3: LLM-as-judge の点数を確率から求める

採点役（`Llama-3.2-1B-Instruct`）が最初に書くトークンが「1」〜「5」のどれかを確率で見て、点数の期待値を返す関数を完成させてください。

1. `judge(**inputs).logits[0, -1]` が、次に書くトークンのスコア（ロジット）
2. その中から「1」〜「5」のトークン ID（`SCORE_IDS`）の 5 つだけを取り出し、`softmax` で確率にする
3. 期待値 $= \sum_{s=1}^{5} s \times P(s)$ を返す

In [ ]:
# 採点役の読み込み（完成済み）
JUDGE_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

judge_tok = AutoTokenizer.from_pretrained(JUDGE_NAME, clean_up_tokenization_spaces=False)
judge = AutoModelForCausalLM.from_pretrained(
    JUDGE_NAME, dtype=torch.float16 if device == 'cuda' else torch.float32, device_map='auto',
)
judge.eval()
SCORE_IDS = [judge_tok.encode(str(s), add_special_tokens=False)[0] for s in range(1, 6)]   # 「1」〜「5」のトークン ID
print(f'採点役: {JUDGE_NAME}')

JUDGE_TEMPLATE = """次の【回答】を、【模範解答】と比べて 1〜5 点で採点してください。
5 = 同じ内容が正確に書かれている、3 = 半分くらい合っている、1 = 間違っている

【模範解答】
{reference}

【回答】
{answer}

点数を 1〜5 の数字 1 文字だけで答えてください。"""


def judge_probs(reference: str, answer: str) -> torch.Tensor:
    messages = [{'role': 'user', 'content': JUDGE_TEMPLATE.format(reference=reference, answer=answer)}]
    inputs = judge_tok.apply_chat_template(messages, add_generation_prompt=True,
                                           return_tensors='pt', return_dict=True).to(judge.device)
    with torch.no_grad():
        logits = judge(**inputs).logits[0, -1].float()
    # TODO: 「1」〜「5」のロジットだけを取り出し、softmax で確率にして返してください
    return None


def judge_score(reference: str, answer: str) -> float:
    probs = judge_probs(reference, answer)
    # TODO: 点数の期待値を返してください
    return None


judge_scores = {name: judge_score(reference, h) for name, h in hypotheses.items()}
for name in names:
    print(f'{name:<6}: ROUGE-L = {rouge_scores[name]:.3f}  BERTScore = {bert_scores[name]:.3f}  LLM-as-judge = {judge_scores[name]:.2f}')

# ── 自己チェック ──
p = judge_probs(reference, hypotheses['不完全'])
assert p.shape == (5,), '確率は 5 つ（1〜5 点）のはずです'
assert abs(p.sum().item() - 1) < 1e-4, '確率の合計が 1 になっていません'
assert abs(judge_scores['不完全'] - float((p * torch.arange(1, 6, device=p.device)).sum())) < 1e-4
assert all(1 <= v <= 5 for v in judge_scores.values())
print('✅ OK')

---

## 課題 4: 指標の弱点を見つける

「正反対」の回答は、意味が模範解答と逆です。理想の指標なら、「言い換え」より低い点を付けるはずです。

各指標について、「正反対」に「言い換え」**以上** の点を付けてしまった指標の名前を、リスト `fooled` に入れてください。

In [ ]:
all_scores = {'ROUGE-L': rouge_scores, 'BERTScore': bert_scores, 'LLM-as-judge': judge_scores}

# TODO: 「正反対」の点が「言い換え」の点以上になった指標の名前を集めてください
fooled = None

print(f'「正反対」にだまされた指標: {fooled}')

# ── 自己チェック ──
assert fooled == [m for m, s in all_scores.items() if s['正反対'] >= s['言い換え']]
assert 'ROUGE-L' in fooled, 'ROUGE-L は文字の一致だけを見るので、「正反対」に高い点を付けるはずです'
print('✅ OK')

---

## 考察（任意）

1. 「言い換え」の ROUGE-L と BERTScore を比べてください。意味が同じことを、どちらの指標がよく捉えていますか？
2. 「正反対」を見抜けたのはどの指標でしたか？ 見抜けなかった指標は、何を測っているからだまされたのでしょうか？
3. 採点役を 8B のモデルに替えたら、LLM-as-judge の結果は変わると思いますか？ 変わるとしたら、なぜでしょうか？